# Regressão Logística com Normalização MinMax e Balanceamento Undersampling

In [1]:
from pandas import read_csv

# Carregando os dados
arquivo = 'dados_dataset/dados_limpos.csv'
dados = read_csv(arquivo)
print("dataset carregado")

dataset carregado


## Balanceamento Undersampling

In [2]:
from sklearn import preprocessing
import pandas as pd

# 1. Transformar a coluna Attrition (0 e 1) em um formato que o LabelEncoder reconhece 
label_encoder = preprocessing.LabelEncoder()
dados['Attrition'] = label_encoder.fit_transform(dados['Attrition'])

# 2. Contagem das classes
count_class_0, count_class_1 = dados.Attrition.value_counts()

# 3. Separando as classes
target_class_0 = dados[dados['Attrition'] == 0]
target_class_1 = dados[dados['Attrition'] == 1]

# 4. Realizando o undersampling na classe majoritária (0)
dados_class_0_under = target_class_0.sample(count_class_1)

# 5. Combinando as classes 
dados_under = pd.concat([dados_class_0_under, target_class_1], axis=0)

# Verificação da nova distribuição
print('Random undersampling:')
print(dados_under.Attrition.value_counts())

Random undersampling:
Attrition
0    237
1    237
Name: count, dtype: int64


## Normalização MinMax

In [3]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# 1. Separando os dados 
X = dados_under.drop('Attrition', axis=1).values
Y = dados_under['Attrition'].values

# 2. Divisão 
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# 3. Padronização 
scaler = MinMaxScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Tamanho do X_train após balanceamento:", X_train_scaled.shape)
print("Dados de Treino Padronizados (5 primeiras linhas):\n", X_train_scaled[0:5,:])

Tamanho do X_train após balanceamento: (379, 44)
Dados de Treino Padronizados (5 primeiras linhas):
 [[0.625      0.15136298 0.71428571 0.5        1.         0.44285714
  0.66666667 0.         1.         0.0623015  0.52861452 0.77777778
  0.64285714 1.         0.         0.33333333 0.2        0.16666667
  0.66666667 0.075      0.11111111 0.06666667 0.11764706 1.
  0.         1.         0.         0.         0.         1.
  0.         0.         1.         0.         1.         0.
  0.         0.         0.         0.         0.         1.
  0.         0.        ]
 [0.325      0.18436155 0.60714286 1.         1.         0.84285714
  1.         0.         0.         0.09924836 0.78025337 0.
  0.42857143 0.         0.66666667 0.         0.05       0.66666667
  0.66666667 0.025      0.         0.         0.         0.
  1.         0.         0.         0.         0.         0.
  0.         0.         1.         1.         0.         0.
  0.         0.         0.         0.         0.      

## Regressão Logística

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Instancia o modelo
modelo = LogisticRegression()

# Treina usando os dados padronizados
modelo.fit(X_train_scaled, y_train)

# Faz a previsão no conjunto de teste 
X_test_scaled = scaler.transform(X_test)
y_pred = modelo.predict(X_test_scaled)

# Avalia
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.83      0.83        47
           1       0.83      0.83      0.83        48

    accuracy                           0.83        95
   macro avg       0.83      0.83      0.83        95
weighted avg       0.83      0.83      0.83        95



In [5]:
import csv
import os
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# tempo de predição
inicio = time.time()
y_pred = modelo.predict(X_test_scaled)
fim = time.time()

# Calcular as métricas
acuracia = accuracy_score(y_test, y_pred)
precisao = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
especificidade = tn / (tn + fp)

tempo = fim - inicio

# SALVANDO RESULTADOS

filename = 'resultados_experimentos.csv'

# Nomes das colunas
fields = ['Modelo', 'Descrição', 'Acurácia', 'Precisão', 'Recall', 'Specificity', 'F1 Score', 'Tempo em Segundo'] 
    
rows = [[
    'Regressão Logística', 
    'normalizacao minmax, balanceamento under', 
    f"{acuracia:.8f}", 
    f"{precisao:.8f}", 
    f"{recall:.8f}", 
    f"{especificidade:.8f}", 
    f"{f1:.8f}", 
    f"{tempo:.8f}"
]]


file_exists = os.path.isfile(filename)

with open(filename, 'a', newline='', encoding='utf-8') as f:
    
    write = csv.writer(f)
    
    if not file_exists:
        write.writerow(fields) 
        
    # Escreve a linha com os resultados do modelo
    write.writerows(rows)
    
    f.close()

print(f"Resultados salvos com sucesso em {filename}")

Resultados salvos com sucesso em resultados_experimentos.csv
